## 02 — Extração de entidades com LLMs/SLMs open-source

Este notebook parte do arquivo limpo gerado em `01-clean-data.ipynb` e executa uma etapa de extração de entidades sem depender de dicionários prévios de empresas, municípios ou setores.

A ideia é comparar abordagens abertas: GLiNER como SLM/zero-shot NER, LLMs locais via Ollama e LLMs/SLMs via Hugging Face Transformers. A saída será consolidada por concordância entre modelos e usada nas próximas etapas: tópicos, eventos, resolução de entidades e grafo no Neo4j.

## 1. Setup

In [24]:
# Instalação opcional
# !pip install pandas numpy tqdm python-dotenv pydantic rapidfuzz
# !pip install gliner
# !pip install transformers accelerate torch sentencepiece bitsandbytes
# # Para GPU CUDA no Colab, normalmente o torch já vem instalado. Em ambiente local, instale conforme seu CUDA.
# !pip install ollama

In [25]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 240)

## 1.1. Detecção de GPU/CPU

Esta célula tenta usar GPU quando disponível. Caso `torch.cuda.is_available()` retorne `False`, o notebook segue em CPU.


In [ ]:

def detect_device(prefer_gpu: bool = True):
    """Detecta CUDA/MPS/CPU sem quebrar o notebook quando torch não estiver instalado."""
    try:
        import torch
        if prefer_gpu and torch.cuda.is_available():
            device = 'cuda'
            device_name = torch.cuda.get_device_name(0)
            dtype = torch.float16
            device_map = 'auto'
        elif prefer_gpu and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            device = 'mps'
            device_name = 'Apple Silicon MPS'
            dtype = torch.float32
            device_map = None
        else:
            device = 'cpu'
            device_name = 'CPU'
            dtype = torch.float32
            device_map = None
        return torch, device, device_name, dtype, device_map
    except Exception as e:
        print('PyTorch não disponível ou falhou ao detectar acelerador. Usando CPU.')
        print(repr(e))
        return None, 'cpu', 'CPU', None, None

TORCH, DEVICE, DEVICE_NAME, TORCH_DTYPE, DEVICE_MAP = detect_device(prefer_gpu=True)
print(f'Dispositivo selecionado: {DEVICE} ({DEVICE_NAME})')


## 2. Caminhos de entrada e saída

Este notebook usa como base o arquivo produzido pelo cleaning:

`data/nip_pipeline_outputs/processed/investimento_2016_2024_clean_step01.csv`

In [26]:
DATA_DIR = Path('data')
OUTPUT_DIR = DATA_DIR / 'nip_pipeline_outputs'
PROCESSED_DIR = OUTPUT_DIR / 'processed'
NER_DIR = OUTPUT_DIR / 'ner'
REPORTS_DIR = OUTPUT_DIR / 'reports'

NER_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CLEAN_FILE = PROCESSED_DIR / 'investimento_2016_2024_clean_step01.csv'

OUTPUT_ENTITY_MENTIONS = NER_DIR / 'entity_mentions_llm_step02.csv'
OUTPUT_CONSENSUS = NER_DIR / 'entity_mentions_consensus_step02.csv'
OUTPUT_RAW_JSONL = NER_DIR / 'ner_llm_raw_outputs_step02.jsonl'
OUTPUT_REPORT = REPORTS_DIR / 'summary_ner_step02.json'

INPUT_CLEAN_FILE

WindowsPath('data/nip_pipeline_outputs/processed/investimento_2016_2024_clean_step01.csv')

## 3. Carregar base limpa

In [27]:
if not INPUT_CLEAN_FILE.exists():
    raise FileNotFoundError(
        f'Arquivo não encontrado: {INPUT_CLEAN_FILE}. Execute primeiro o notebook 01-clean-data.ipynb.'
    )

df = pd.read_csv(INPUT_CLEAN_FILE)
print(f'Linhas: {len(df):,}')
print(f'Colunas: {len(df.columns):,}')
display(df.head(3))

Linhas: 15,211
Colunas: 12


,noticia_id,data_publicacao,titulo,titulo_limpo,fonte,texto,texto_limpo,texto_base,flagnoticia,titulo_len,texto_len,texto_base_len
0,5c2f55d84fa33a4521c84952,2016-01-01 00:00:00,Araçoiaba busca melhorar abastecimento de água,Araçoiaba busca melhorar abastecimento de água,Diário de Sorocaba,"Para atender a demanda de crescimento dos bairros abastecidos pelo Reservatório Suíço, a concessionária Águas de Araçoiaba finalizou, neste mês de dezembro, melhorias no sistema de bombeamento da unidade. As obras tiveram início em outu...","Para atender a demanda de crescimento dos bairros abastecidos pelo Reservatório Suíço, a concessionária Águas de Araçoiaba finalizou, neste mês de dezembro, melhorias no sistema de bombeamento da unidade. As obras tiveram início em outu...","Araçoiaba busca melhorar abastecimento de água\nPara atender a demanda de crescimento dos bairros abastecidos pelo Reservatório Suíço, a concessionária Águas de Araçoiaba finalizou, neste mês de dezembro, melhorias no sistema de bombeam...",I,46,1580,1627
1,265e1fc4bd247c649173c83d,2016-01-02 00:00:00,Franquias da região conquistam mercado local e internacional,Franquias da região conquistam mercado local e internacional,Folha da Região,"Quem caminha por ruas comerciais e pelos centros de compras na região avista um grande número de fachadas de franquias, a maioria importada de outras partes do Brasil. Mas há redes de franchising que fazem o caminho inverso. Algumas emp...","Quem caminha por ruas comerciais e pelos centros de compras na região avista um grande número de fachadas de franquias, a maioria importada de outras partes do Brasil. Mas há redes de franchising que fazem o caminho inverso. Algumas emp...","Franquias da região conquistam mercado local e internacional\nQuem caminha por ruas comerciais e pelos centros de compras na região avista um grande número de fachadas de franquias, a maioria importada de outras partes do Brasil. Mas há...",I,60,3027,3088
2,72519e6106b0f258b1880485,2016-01-02 00:00:00,"Hospital da Criança, do Grendac, será entregue até março deste ano","Hospital da Criança, do Grendac, será entregue até março deste ano",Jornal de Jundiaí,"Obras do Hospital da Criança, que vai atender diversas especialidades médicas além da oncologia, estão dentro do cronograma\nA expectativa dos administradores do Grupo em Defesa da Criança com Câncer (Grendacc), em Jundiaí, é de que o H...","Obras do Hospital da Criança, que vai atender diversas especialidades médicas além da oncologia, estão dentro do cronograma\nA expectativa dos administradores do Grupo em Defesa da Criança com Câncer (Grendacc), em Jundiaí, é de que o H...","Hospital da Criança, do Grendac, será entregue até março deste ano\nObras do Hospital da Criança, que vai atender diversas especialidades médicas além da oncologia, estão dentro do cronograma\nA expectativa dos administradores do Grupo ...",I,66,2384,2451


In [28]:
required_columns = {'noticia_id', 'data_publicacao', 'titulo', 'fonte', 'texto_base'}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f'Colunas obrigatórias ausentes: {missing}')

if 'flagnoticia' in df.columns:
    df = df[df['flagnoticia'].astype(str).str.upper().eq('I')].copy()

print(f'Linhas após garantir flag I: {len(df):,}')
print(f'IDs únicos: {df["noticia_id"].nunique():,}')

Linhas após garantir flag I: 15,211
IDs únicos: 15,211


## 4. Amostragem inicial

In [29]:
RANDOM_STATE = 42
N_SAMPLE = 30  # altere para None para rodar no dataset inteiro
MAX_CHARS_PER_NEWS = 4500

if N_SAMPLE is None:
    df_work = df.copy()
else:
    df_work = df.sample(n=min(N_SAMPLE, len(df)), random_state=RANDOM_STATE).copy()

df_work['texto_para_ner'] = df_work['texto_base'].fillna('').astype(str).str.slice(0, MAX_CHARS_PER_NEWS)

print(f'Notícias a processar nesta rodada: {len(df_work):,}')
display(df_work[['noticia_id', 'data_publicacao', 'fonte', 'titulo', 'texto_para_ner']].head(3))

Notícias a processar nesta rodada: 30


,noticia_id,data_publicacao,fonte,titulo,texto_para_ner
1087,edbc9ea00c78a0e41fd1c932,2016-09-22 00:00:00,Diário do Grande ABC,Mercedes-Benz anuncia pista de testes no Brasil,"Mercedes-Benz anuncia pista de testes no Brasil\nEm fase de ampla reestruturação de suas operações em São Bernardo do Campo, no ABC paulista, para reduzir uma ociosidade produtiva que passa de 50%, a Mercedes- Benz do Brasil anunciou na..."
10052,eac0ccdd3674d79b6db2c364,2020-10-21 00:00:00,Diário da Região/São José do Rio Preto,Bares e restaurantes investem em tecnologia para cumprir protocolos sanitários,"Bares e restaurantes investem em tecnologia para cumprir protocolos sanitários\nCom a reabertura dos bares, lanchonetes e restaurantes em Rio Preto, uma série de soluções inovadoras tem sido adotada pelos comerciantes do setor para cump..."
8722,06a62008f1216ed69677c33b,2019-11-26 00:00:00,Diário do Grande ABC,S.Caetano terá área dedicada a motoristas de app,S.Caetano terá área dedicada a motoristas de app\n26/11/2019 | 07:00\nSão Caetano vai acolher a primeira loja do País de aluguel de carros exclusivo para uso de aplicativos de transporte de passageiros.\nA Movida inaugura hoje espaço de...


## 5. Utilitários de parsing, normalização e validação

In [30]:
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r'[^a-z0-9\s\$\.,%-]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def extract_json_object(text: str):
    if text is None:
        return None
    text = str(text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find('{')
    end = text.rfind('}')
    if start >= 0 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            return None
    return None


def safe_entities_from_response(response_text: str):
    obj = extract_json_object(response_text)
    if not isinstance(obj, dict):
        return []
    entities = obj.get('entities', [])
    if not isinstance(entities, list):
        return []
    clean = []
    for ent in entities:
        if not isinstance(ent, dict):
            continue
        text = str(ent.get('text', '')).strip()
        if not text:
            continue
        clean.append({
            'text': text,
            'type': str(ent.get('type', 'entidade')).strip(),
            'role': str(ent.get('role', '')).strip(),
            'evidence': str(ent.get('evidence', '')).strip(),
            'confidence': ent.get('confidence', None),
        })
    return clean

## 6. Prompt de extração open-schema

O objetivo é evitar listas fechadas de empresas, municípios ou setores. O modelo identifica entidades relevantes para representar a notícia em um grafo de conhecimento.

In [31]:
def build_open_schema_prompt(title: str, text: str) -> str:
    template = [
        'Você é um sistema de extração de entidades para construção de um grafo de conhecimento sobre notícias de investimentos produtivos.',
        '',
        'Extraia entidades relevantes do texto. Não use conhecimento externo. Extraia apenas entidades explicitamente mencionadas.',
        'Escolha tipos de entidade livremente, de forma curta e consistente. Exemplos possíveis, mas não obrigatórios: empresa, organização, localidade, valor monetário, data, setor econômico, produto, serviço, unidade produtiva, fonte, pessoa.',
        '',
        'Regras:',
        '- Retorne somente JSON válido.',
        '- Não invente entidades.',
        '- Use a menção como aparece no texto.',
        '- Inclua um trecho de evidência curto.',
        '- Se não houver entidades, retorne lista vazia.',
        '',
        'Formato obrigatório:',
        '{',
        '  "entities": [',
        '    {',
        '      "text": "menção literal",',
        '      "type": "tipo curto escolhido pelo modelo",',
        '      "role": "papel da entidade no contexto da notícia",',
        '      "evidence": "trecho curto do texto que justifica",',
        '      "confidence": 0.0',
        '    }',
        '  ]',
        '}',
        '',
        'TÍTULO:',
        str(title),
        '',
        'TEXTO:',
        str(text),
    ]
    return '\n'.join(template)

## 7. Método A — GLiNER como SLM zero-shot para NER

In [32]:
USE_GLINER = True
GLINER_MODEL_NAME = 'urchade/gliner_multi-v2.1'

# Rótulos amplos, voltados ao grafo. Não são listas de entidades.
GLINER_LABELS = [
    'empresa',
    'organização',
    'localidade',
    'valor monetário',
    'data',
    'setor econômico',
    'produto ou serviço',
    'unidade produtiva',
    'pessoa'
]

In [33]:

gliner_model = None

if USE_GLINER:
    try:
        from gliner import GLiNER
        gliner_model = GLiNER.from_pretrained(GLINER_MODEL_NAME)

        # Move o modelo para GPU quando disponível; se falhar, volta para CPU.
        if hasattr(gliner_model, 'to'):
            try:
                gliner_model = gliner_model.to(DEVICE)
            except Exception as gpu_error:
                print(f'Não foi possível mover GLiNER para {DEVICE}. Tentando CPU.')
                print(repr(gpu_error))
                DEVICE = 'cpu'
                gliner_model = gliner_model.to('cpu')

        print(f'GLiNER carregado: {GLINER_MODEL_NAME}')
        print(f'GLiNER executando em: {DEVICE}')
    except Exception as e:
        USE_GLINER = False
        print('Não foi possível carregar GLiNER. A etapa será ignorada.')
        print(repr(e))


c:\Users\joaov\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 5 files: 100%|██████████| 5/5 [00:00<?, ?it/s]


GLiNER carregado: urchade/gliner_multi-v2.1


In [34]:
def extract_with_gliner(row, threshold=0.35):
    if gliner_model is None:
        return []
    text = row['texto_para_ner']
    preds = gliner_model.predict_entities(text, GLINER_LABELS, threshold=threshold)
    entities = []
    for p in preds:
        entities.append({
            'noticia_id': row['noticia_id'],
            'model': GLINER_MODEL_NAME,
            'method': 'gliner_zero_shot',
            'text': p.get('text', ''),
            'type': p.get('label', ''),
            'role': '',
            'evidence': p.get('text', ''),
            'confidence': p.get('score', None),
            'start': p.get('start', None),
            'end': p.get('end', None),
        })
    return entities

all_mentions = []

if USE_GLINER:
    for _, row in tqdm(df_work.iterrows(), total=len(df_work), desc='GLiNER'):
        all_mentions.extend(extract_with_gliner(row))

print(f'Menções GLiNER: {len(all_mentions):,}')

GLiNER:   0%|          | 0/30 [00:00<?, ?it/s]c:\Users\joaov\AppData\Local\Programs\Python\Python312\Lib\site-packages\gliner\data_processing\processor.py:395: UserWarning: Sentence of length 572 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
GLiNER:   3%|▎         | 1/30 [00:01<00:34,  1.18s/it]c:\Users\joaov\AppData\Local\Programs\Python\Python312\Lib\site-packages\gliner\data_processing\processor.py:395: UserWarning: Sentence of length 705 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
GLiNER:   7%|▋         | 2/30 [00:01<00:25,  1.09it/s]c:\Users\joaov\AppData\Local\Programs\Python\Python312\Lib\site-packages\gliner\data_processing\processor.py:395: UserWarning: Sentence of length 410 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
GLiNER:  13%|█▎      

Menções GLiNER: 456


## 8. Método B — LLMs locais via Ollama

Exemplos de modelos:

```bash
ollama pull llama3.1:8b
ollama pull qwen2.5:7b-instruct
ollama pull gemma2:9b
```

In [35]:
USE_OLLAMA = True  # altere para True se estiver com Ollama instalado e rodando
OLLAMA_MODELS = [
    'llama3.1:8b',
    'qwen2.5:7b-instruct',
    'gemma2:9b',
]

In [36]:
def extract_with_ollama(row, model_name: str):
    import ollama
    prompt = build_open_schema_prompt(row['titulo'], row['texto_para_ner'])
    response = ollama.chat(
        model=model_name,
        messages=[
            {'role': 'system', 'content': 'Você extrai entidades de notícias em português e responde somente JSON válido.'},
            {'role': 'user', 'content': prompt},
        ],
        options={'temperature': 0}
    )
    content = response['message']['content']
    entities = safe_entities_from_response(content)
    rows = []
    for ent in entities:
        rows.append({
            'noticia_id': row['noticia_id'],
            'model': model_name,
            'method': 'ollama_llm_json',
            'text': ent['text'],
            'type': ent['type'],
            'role': ent['role'],
            'evidence': ent['evidence'],
            'confidence': ent['confidence'],
            'start': None,
            'end': None,
        })
    raw = {
        'noticia_id': row['noticia_id'],
        'model': model_name,
        'method': 'ollama_llm_json',
        'raw_response': content,
        'parsed_entities_count': len(rows),
    }
    return rows, raw

In [37]:
raw_outputs = []

if USE_OLLAMA:
    for model_name in OLLAMA_MODELS:
        for _, row in tqdm(df_work.iterrows(), total=len(df_work), desc=f'Ollama {model_name}'):
            try:
                rows, raw = extract_with_ollama(row, model_name)
                all_mentions.extend(rows)
                raw_outputs.append(raw)
            except Exception as e:
                raw_outputs.append({
                    'noticia_id': row['noticia_id'],
                    'model': model_name,
                    'method': 'ollama_llm_json',
                    'error': repr(e),
                })

print(f'Menções acumuladas: {len(all_mentions):,}')
print(f'Respostas brutas Ollama: {len(raw_outputs):,}')

Ollama gemma2:9b: 100%|██████████| 30/30 [01:00<00:00,  2.03s/it]

Menções acumuladas: 456
Respostas brutas Ollama: 90


## 9. Método C — LLMs/SLMs open-source via Hugging Face Transformers

In [38]:
USE_HF_GENERATIVE = True
HF_MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

In [39]:

hf_pipeline = None
hf_tokenizer = None
hf_model = None

if USE_HF_GENERATIVE:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

        hf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)

        model_kwargs = {}
        if TORCH_DTYPE is not None:
            model_kwargs['torch_dtype'] = TORCH_DTYPE
        if DEVICE == 'cuda':
            model_kwargs['device_map'] = 'auto'

        hf_model = AutoModelForCausalLM.from_pretrained(
            HF_MODEL_NAME,
            **model_kwargs,
        )

        # Se não usamos device_map='auto', movemos manualmente para o dispositivo escolhido.
        if DEVICE != 'cuda' and hasattr(hf_model, 'to'):
            try:
                hf_model = hf_model.to(DEVICE)
            except Exception as move_error:
                print(f'Não foi possível mover o modelo HF para {DEVICE}. Tentando CPU.')
                print(repr(move_error))
                DEVICE = 'cpu'
                hf_model = hf_model.to('cpu')

        hf_pipeline = pipeline(
            'text-generation',
            model=hf_model,
            tokenizer=hf_tokenizer,
            max_new_tokens=700,
            do_sample=False,
            temperature=0.0,
        )
        print(f'Modelo HF carregado: {HF_MODEL_NAME}')
        print(f'Modelo HF executando em: {DEVICE}')
    except Exception as e:
        USE_HF_GENERATIVE = False
        print('Não foi possível carregar modelo HF generativo.')
        print(repr(e))


c:\Users\joaov\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joaov\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 434/434 [00:00<00:00, 1067.72it/s, Materializing param=

Modelo HF carregado: Qwen/Qwen2.5-3B-Instruct


In [40]:
def extract_with_hf_generative(row):
    if hf_pipeline is None:
        return [], None

    prompt = build_open_schema_prompt(row['titulo'], row['texto_para_ner'])

    messages = [
        {'role': 'system', 'content': 'Você extrai entidades de notícias em português e responde somente JSON válido.'},
        {'role': 'user', 'content': prompt},
    ]

    try:
        input_text = hf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        input_text = prompt

    out = hf_pipeline(input_text)[0]['generated_text']
    content = out[len(input_text):].strip() if out.startswith(input_text) else out

    entities = safe_entities_from_response(content)
    rows = []
    for ent in entities:
        rows.append({
            'noticia_id': row['noticia_id'],
            'model': HF_MODEL_NAME,
            'method': 'hf_generative_json',
            'text': ent['text'],
            'type': ent['type'],
            'role': ent['role'],
            'evidence': ent['evidence'],
            'confidence': ent['confidence'],
            'start': None,
            'end': None,
        })
    raw = {
        'noticia_id': row['noticia_id'],
        'model': HF_MODEL_NAME,
        'method': 'hf_generative_json',
        'raw_response': content,
        'parsed_entities_count': len(rows),
    }
    return rows, raw

In [ ]:
if USE_HF_GENERATIVE:
    for _, row in tqdm(df_work.iterrows(), total=len(df_work), desc=f'HF {HF_MODEL_NAME}'):
        try:
            rows, raw = extract_with_hf_generative(row)
            all_mentions.extend(rows)
            raw_outputs.append(raw)
        except Exception as e:
            raw_outputs.append({
                'noticia_id': row['noticia_id'],
                'model': HF_MODEL_NAME,
                'method': 'hf_generative_json',
                'error': repr(e),
            })

print(f'Menções acumuladas: {len(all_mentions):,}')

HF Qwen/Qwen2.5-3B-Instruct:   0%|          | 0/30 [00:00<?, ?it/s]Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 10. Consolidar menções extraídas

In [ ]:
mentions_df = pd.DataFrame(all_mentions)

if mentions_df.empty:
    print('Nenhuma menção extraída. Ative pelo menos um método ou verifique dependências/modelos.')
else:
    mentions_df['text_norm'] = mentions_df['text'].apply(normalize_text)
    mentions_df['type_norm'] = mentions_df['type'].apply(normalize_text)
    mentions_df = mentions_df[mentions_df['text_norm'].str.len() > 1].copy()
    mentions_df = mentions_df.drop_duplicates(
        subset=['noticia_id', 'model', 'method', 'text_norm', 'type_norm']
    ).reset_index(drop=True)

    display(mentions_df.head(20))
    print(f'Total de menções: {len(mentions_df):,}')
    print(f'Notícias com pelo menos uma menção: {mentions_df["noticia_id"].nunique():,}')

,noticia_id,model,method,text,type,role,evidence,confidence,start,end,text_norm,type_norm
0,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,Mercedes-Benz,empresa,,Mercedes-Benz,0.702720,0,13,mercedes-benz,empresa
1,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,São Bernardo do Campo,localidade,,São Bernardo do Campo,0.736342,101,122,sao bernardo do campo,localidade
2,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,ABC paulista,localidade,,ABC paulista,0.609290,127,139,abc paulista,localidade
3,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,Mercedes- Benz do Brasil,unidade produtiva,,Mercedes- Benz do Brasil,0.514049,199,223,mercedes- benz do brasil,unidade produtiva
4,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,"quarta-feira, 21",data,,"quarta-feira, 21",0.587878,236,252,"quarta-feira, 21",data
5,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,Iracemápolis,localidade,,Iracemápolis,0.701342,628,640,iracemapolis,localidade
6,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,SP,localidade,,SP,0.412937,642,644,sp,localidade
7,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,Mercedes do Brasil,unidade produtiva,,Mercedes do Brasil,0.375894,830,848,mercedes do brasil,unidade produtiva
8,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,Philipp Schiemer,pessoa,,Philipp Schiemer,0.927824,850,866,philipp schiemer,pessoa
9,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,R$ 70 milhões,valor monetário,,R$ 70 milhões,0.882734,902,915,r$ 70 milhoes,valor monetario


Total de menções: 330
Notícias com pelo menos uma menção: 30


## 11. Consolidação por concordância entre modelos

In [ ]:
if not mentions_df.empty:
    consensus = (
        mentions_df
        .groupby(['noticia_id', 'text_norm'], as_index=False)
        .agg(
            text=('text', 'first'),
            types=('type', lambda x: sorted(set(map(str, x)))),
            methods=('method', lambda x: sorted(set(map(str, x)))),
            models=('model', lambda x: sorted(set(map(str, x)))),
            n_methods=('method', 'nunique'),
            n_models=('model', 'nunique'),
            max_confidence=('confidence', lambda x: pd.to_numeric(x, errors='coerce').max()),
            examples_evidence=('evidence', lambda x: ' | '.join([str(v) for v in list(x)[:3] if str(v).strip()])),
        )
        .sort_values(['n_models', 'n_methods', 'noticia_id'], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    display(consensus.head(30))
else:
    consensus = pd.DataFrame()

,noticia_id,text_norm,text,types,methods,models,n_methods,n_models,max_confidence,examples_evidence
0,05af7d37a8d6195eb55892d0,09 de novembro de 2018,09 de Novembro de 2018,[data],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.480501,09 de Novembro de 2018
1,05af7d37a8d6195eb55892d0,2015,2015,[data],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.383532,2015
2,05af7d37a8d6195eb55892d0,espirito santo,Espírito Santo,[localidade],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.689127,Espírito Santo
3,05af7d37a8d6195eb55892d0,hortifruti,Hortifruti,[empresa],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.507536,Hortifruti
4,05af7d37a8d6195eb55892d0,natural da terra,Natural da Terra,[empresa],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.510067,Natural da Terra
5,05af7d37a8d6195eb55892d0,novembro de 2015,novembro de 2015,[data],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.578325,novembro de 2015
6,05af7d37a8d6195eb55892d0,quinta-feira,quinta-feira,[data],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.362362,quinta-feira
7,05af7d37a8d6195eb55892d0,r$ 300 milhoes,R$ 300 milhões,[valor monetário],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.937337,R$ 300 milhões
8,05af7d37a8d6195eb55892d0,r$ 80 milhoes,R$ 80 milhões,[valor monetário],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.929663,R$ 80 milhões
9,05af7d37a8d6195eb55892d0,rio de janeiro,Rio de Janeiro,[localidade],[gliner_zero_shot],[urchade/gliner_multi-v2.1],1,1,0.851373,Rio de Janeiro


## 12. Exportar resultados

In [ ]:
# Salva menções e consenso
if not mentions_df.empty:
    mentions_df.to_csv(OUTPUT_ENTITY_MENTIONS, index=False, encoding="utf-8")
    consensus.to_csv(OUTPUT_CONSENSUS, index=False, encoding="utf-8")
else:
    # Cria arquivos vazios com colunas esperadas, para manter o pipeline reprodutível
    pd.DataFrame(columns=[
        "noticia_id",
        "method",
        "entity_text",
        "entity_type",
        "role",
        "evidence",
        "confidence",
    ]).to_csv(OUTPUT_ENTITY_MENTIONS, index=False, encoding="utf-8")

    pd.DataFrame(columns=[
        "noticia_id",
        "entity_text_norm",
        "entity_type",
        "n_methods",
        "methods",
    ]).to_csv(OUTPUT_CONSENSUS, index=False, encoding="utf-8")

# Salva saídas brutas em JSONL: um JSON por linha
with open(OUTPUT_RAW_JSONL, "w", encoding="utf-8") as f:
    for item in raw_outputs:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

summary = {
    "created_at": datetime.now().isoformat(),
    "input_clean_file": str(INPUT_CLEAN_FILE),
    "n_news_available": int(len(df)),
    "n_news_processed": int(len(df_work)),
    "n_mentions": int(len(mentions_df)) if not mentions_df.empty else 0,
    "n_news_with_mentions": int(mentions_df["noticia_id"].nunique()) if not mentions_df.empty else 0,
    "device": DEVICE,
    "device_name": DEVICE_NAME,
    "methods_enabled": {
        "gliner": bool(USE_GLINER),
        "ollama": bool(USE_OLLAMA),
        "hf_generative": bool(USE_HF_GENERATIVE),
    },
    "output_entity_mentions": str(OUTPUT_ENTITY_MENTIONS),
    "output_consensus": str(OUTPUT_CONSENSUS),
    "output_raw_jsonl": str(OUTPUT_RAW_JSONL),
}

with open(OUTPUT_REPORT, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

,value
created_at,2026-05-26T02:30:04.313928
input_clean_file,data\nip_pipeline_outputs\processed\investimento_2016_2024_clean_step01.csv
n_news_available,15211
n_news_processed,30
n_mentions,330
n_news_with_mentions,30
methods_enabled,"{'gliner': True, 'ollama': False, 'hf_generative': False}"
output_entity_mentions,data\nip_pipeline_outputs\ner\entity_mentions_llm_step02.csv
output_consensus,data\nip_pipeline_outputs\ner\entity_mentions_consensus_step02.csv
output_raw_jsonl,data\nip_pipeline_outputs\ner\ner_llm_raw_outputs_step02.jsonl


## 13. Amostra para avaliação manual

In [ ]:
if not mentions_df.empty:
    review_sample = mentions_df.sample(n=min(100, len(mentions_df)), random_state=RANDOM_STATE).copy()
    review_sample['manual_label'] = ''
    review_sample['manual_notes'] = ''
    review_file = NER_DIR / 'entity_mentions_manual_review_sample_step02.csv'
    review_sample.to_csv(review_file, index=False, encoding='utf-8')
    print(f'Amostra para revisão salva em: {review_file}')
    display(review_sample.head(20))

Amostra para revisão salva em: data\nip_pipeline_outputs\ner\entity_mentions_manual_review_sample_step02.csv


,noticia_id,model,method,text,type,role,evidence,confidence,start,end,text_norm,type_norm,manual_label,manual_notes
9,edbc9ea00c78a0e41fd1c932,urchade/gliner_multi-v2.1,gliner_zero_shot,R$ 70 milhões,valor monetário,,R$ 70 milhões,0.882734,902,915,r$ 70 milhoes,valor monetario,,
164,7dd241163c8a381f6778e1d1,urchade/gliner_multi-v2.1,gliner_zero_shot,Philip Stanhope,pessoa,,Philip Stanhope,0.374387,920,935,philip stanhope,pessoa,,
139,337ba1a9ef1452b28cb24edd,urchade/gliner_multi-v2.1,gliner_zero_shot,Fernando Collor de Mello,pessoa,,Fernando Collor de Mello,0.791329,71,95,fernando collor de mello,pessoa,,
46,827a0b6556f375d8c7fba29d,urchade/gliner_multi-v2.1,gliner_zero_shot,Coca-Cola,empresa,,Coca-Cola,0.707002,0,9,coca-cola,empresa,,
94,4ffc35c8fda8eb2fbf686836,urchade/gliner_multi-v2.1,gliner_zero_shot,Penápolis,localidade,,Penápolis,0.738150,1470,1479,penapolis,localidade,,
101,4ffc35c8fda8eb2fbf686836,urchade/gliner_multi-v2.1,gliner_zero_shot,Bauru-Arealva,localidade,,Bauru-Arealva,0.714760,1927,1940,bauru-arealva,localidade,,
84,9133f1e62b4449f326465a4f,urchade/gliner_multi-v2.1,gliner_zero_shot,Diadema,localidade,,Diadema,0.741928,1907,1914,diadema,localidade,,
311,28962c7b468b0418f06585c8,urchade/gliner_multi-v2.1,gliner_zero_shot,Secretaria de Estado da Saúde,organização,,Secretaria de Estado da Saúde,0.474781,1195,1224,secretaria de estado da saude,organizacao,,
316,8e78bef7fbff81373811dfb6,urchade/gliner_multi-v2.1,gliner_zero_shot,Brasil,localidade,,Brasil,0.493192,42,48,brasil,localidade,,
219,43243e0bd6458c69e460fb4a,urchade/gliner_multi-v2.1,gliner_zero_shot,América Latina,localidade,,América Latina,0.385739,673,687,america latina,localidade,,
